# ex: 회원가입부터 상품 주문과 삭제까지

**오늘 발표할 내용:** GUI의 입력창과 버튼을 Jupyter 셀로 대신하고, 기존 서비스 클래스를 호출하여 여러 테이블을 연결한다.

기존 `users.ipynb`, `products.ipynb`, `orders.ipynb`, `order_items.ipynb` 파일과 클래스는 수정하지 않는다. 새 GUI, 새 서비스 클래스, 결제, 재고 관리 기능도 추가하지 않는다.

## 실행 순서

1. **준비** 셀을 실행한다. VS Code 커널은 이 저장소의 `.venv`를 선택한다.
2. DB에 판매 상품이 없다면 **선택 단계 S**에서 판매자 계정과 상품을 준비한다. 상품이 있으면 건너뛴다.
3. **1~8단계**를 위에서 아래로 실행한다. 입력값을 바꾸고 셀을 실행하는 것이 버튼 클릭에 해당한다.
4. 주문 직후 `orders`와 `order_items`를 확인하고, 삭제 셀 실행 후 `deleted_at`의 변화를 비교한다.

**실제 DB를 사용하는 예제다.** 기본 회원가입 옵션은 꺼져 있다. 켜서 실행하면 실제 Auth 계정이 생성되고, 주문/삭제 셀은 실제 실습 DB를 변경한다. 비밀번호는 `getpass()`로 입력하며 출력하거나 파일에 저장하지 않는다. 시연 후 노트북 출력을 지우고 저장한다.

작성 시 확인: Supabase 테이블 정의에서 주문의 필수 `order_no`, `order_name`, 기본 UUID/시각, 주문상품의 FK와 가격/수량 조건을 확인했다. `products`, `orders`, `order_items`는 Table Editor에서 비어 있었다. 계정 생성과 DB 쓰기는 자동으로 실행하지 않았다. 과제 링크 본문은 접근되지 않아 사용자가 승인한 테이블 기준으로 구성했다.

> **실제 DB 시연 전 확인할 한 가지:** 2026-09-11 확인한 `products`는 RLS가 켜져 있고 정책이 0개다. 따라서 현재 설정 그대로는 로그인해도 상품 조회/등록이 허용되지 않는다. 상품 담당자가 예제에 필요한 상품 SELECT와 판매자 INSERT 정책을 준비한 뒤 상품 단계를 실행해야 한다. 코드를 설명하는 발표는 바로 가능하지만, 실제 주문 시연은 이 전제가 필요하다. 이번 작업은 DB 정책을 수정하지 않았다. `user_details`에는 개발용 ALL 정책이 있고 `orders`/`order_items`는 RLS가 비활성화되어 있어, 사용자별 보안 정책까지 구현한 예제라고 설명하면 안 된다.

## 준비 A. 연결은 하나만 만든다

Supabase 클라이언트는 서버와 요청을 주고받는 객체다. 이 객체로 로그인하면 이후 DB 요청에도 그 로그인 세션이 사용된다. 로그인한 뒤 다른 클라이언트를 새로 만들면 같은 로그인 상태라고 생각해서는 안 된다.

`.env`는 프로젝트 폴더에 있어야 한다. `SUPABASE_URL`에는 프로젝트 기본 URL을 넣고 `/rest/v1/`을 붙이지 않는다. `SUPABASE_PUBLISHABLE_KEY`에는 프로젝트의 공개 API 키를 넣는다. 관리자 키는 필요 없다.

In [ ]:
import os
import json
import uuid
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal
from getpass import getpass
from pprint import pprint
from dotenv import load_dotenv
from supabase import create_client

# AX_TeamDB 폴더를 열고 실행한다. 경로가 다르면 여기서 먼저 멈춘다.
project_dir = Path.cwd()
if not (project_dir / "users.ipynb").exists():
    raise RuntimeError("AX_TeamDB 폴더에서 이 노트북을 실행하세요.")
load_dotenv(project_dir / ".env", override=True)
url = os.getenv("SUPABASE_URL", "").rstrip("/")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY", "")
if not url or not key or "/rest/v1" in url:
    raise ValueError(".env의 프로젝트 기본 URL과 공개 API 키를 확인하세요.")
supabase = create_client(url, key)
user_id = None  # 연결을 새로 만들었으므로 이전 로그인 ID를 재사용하지 않는다.

# 같은 커널에서 준비 셀을 다시 눌러도 진행 중인 주문 ID는 버리지 않는다.
# 주문 도중 커널을 재시작하면 이 기억도 사라진다. 삭제 전 ID를 확인해 둔다.
demo_order_id = globals().get("demo_order_id")
print("연결 객체 준비 완료. 아직 로그인하거나 데이터를 저장하지 않았습니다.")

## 준비 B. 기존 클래스 정의만 불러온다

`%run products.ipynb`는 파일의 모든 코드 셀을 실행한다. 현재 그 파일에는 독립적인 회원가입/로그인 실습도 있어서 그대로 실행하면 예제에 필요 없는 요청이 발생한다.

아래 준비 코드는 **확인한 로컬 파일에서 `class`로 시작하는 클래스 셀만 실행**한다. 파일 내용이나 클래스 메서드는 바꾸지 않는다. 이 부분은 `.ipynb`를 같이 쓰기 위한 준비일 뿐, 오늘 발표할 핵심 로직은 아니다. 외부에서 받은 모르는 파일에 이 방식을 적용하지 않는다.

`UserService()`처럼 괄호가 있어야 인스턴스를 만든다. `order_service = Order_service`는 클래스 이름을 다른 변수에 담은 것이고, `Order_service(supabase)`가 실제 객체 생성이다. C++의 타입과 생성된 객체를 구분하는 것과 비슷하다.

현재 주문 조회/수정 메서드와 상품 조회/등록 메서드에는 미완성 또는 잘못된 호출이 있다. 이 예제는 완성된 메서드를 재사용하고, 나머지는 `ex` 안에 짧은 SDK 쿼리를 작성한다. 기존 메서드를 교체하거나 덮어쓰지 않는다.

In [ ]:
# 세 파일은 클래스 전체가 하나의 코드 셀에 들어 있음을 확인했다.
for filename, class_name in [
    ("users.ipynb", "UserService"),
    ("orders.ipynb", "Order_service"),
    ("order_items.ipynb", "OrderItemService"),
]:
    notebook = json.loads((project_dir / filename).read_text(encoding="utf-8"))
    class_cells = [
        "".join(cell["source"])
        for cell in notebook["cells"]
        if cell["cell_type"] == "code"
        and "".join(cell["source"]).lstrip().startswith(f"class {class_name}:")
    ]
    if len(class_cells) != 1:
        raise RuntimeError(f"{filename}의 클래스 셀 구성이 달라졌습니다.")
    exec(class_cells[0], globals())

# 기존 일부 메서드는 전역 supabase를 사용하므로 위에서 만든 하나의 객체를 유지한다.
user_service = UserService()
order_service = Order_service(supabase)
order_item_service = OrderItemService(supabase)
print("기존 회원, 주문, 주문상품 클래스 준비 완료")

## 선택 단계 S. 상품이 없을 때만 판매자 계정으로 예제 상품 준비

이미 팀원이 등록한 상품이 있다면 아래 셀은 그대로 건너뛴다. 상품이 없다면 `PREPARE_PRODUCT = True`로 바꾸고 **판매자용 계정**으로 실행한다. 신규 판매자 계정일 때만 `NEW_SELLER_ACCOUNT = True`로 바꾼다. 뒤의 구매자 계정과는 다른 계정을 사용하면 역할을 설명하기 쉽다.

실행 순서는 판매자 가입(선택) → 판매자 로그인 → `user_details`의 SELLER 확인/생성 → `products` 등록 → 로그아웃이다. 기존 BUYER를 SELLER로 덮어쓰지는 않는다.

`products.seller_id`에는 현재 로그인한 판매자의 ID가 들어간다. 회원 역할을 Python에서 검사하는 것은 예제의 흐름을 설명하기 위한 것이다. 실제 접근 권한은 DB의 RLS 정책이 허용해야 한다. 권한 오류가 나면 관리자 키로 바꾸거나 정책을 끄지 말고 팀의 판매자 정책을 확인한다.

실제 FK는 `products.seller_id → user_details.id`이므로 Auth 가입만으로는 충분하지 않고 판매자의 상세 정보 행이 먼저 필요하다. 상품 정의의 `seller_id`에는 랜덤 UUID 기본값이 있지만, 이를 생략하면 실제 회원을 가리키지 못할 수 있으므로 로그인 ID를 명시한다. 현재 상품 정책이 비어 있으므로 위 실행 전제를 먼저 확인한다.

**이 셀을 다시 실행하면 상품이 한 건 더 만들어진다.** 상품 준비가 끝나면 `PREPARE_PRODUCT`와 `NEW_SELLER_ACCOUNT`를 다시 `False`로 돌려 둔다. 여기서 만든 상품은 뒤의 주문 삭제로 지워지지 않는다.

In [ ]:
PREPARE_PRODUCT = False
NEW_SELLER_ACCOUNT = False

if PREPARE_PRODUCT:
    seller_email = input("판매자 이메일: ").strip()
    seller_password = getpass("판매자 비밀번호: ")
    try:
        if NEW_SELLER_ACCOUNT:
            signup = supabase.auth.sign_up({"email": seller_email, "password": seller_password})
            if signup.session is None:
                raise RuntimeError("가입 이메일을 확인한 뒤 NEW_SELLER_ACCOUNT=False로 다시 실행하세요.")
        seller_login = supabase.auth.sign_in_with_password({"email": seller_email, "password": seller_password})
        if seller_login.user is None or seller_login.session is None:
            raise RuntimeError("판매자 로그인에 실패했습니다.")
        seller_id = seller_login.user.id
        seller_details = supabase.table("user_details").select("*").eq("id", seller_id).execute().data
        if not seller_details:
            seller_details = user_service.create_user_detail(seller_id, type="SELLER")
        if not seller_details or seller_details[0]["type"] != "SELLER" or seller_details[0]["deleted_at"] is not None:
            raise PermissionError("활성 SELLER 계정이 필요합니다. 기존 회원 유형은 변경하지 않습니다.")

        # 기존 상품 클래스는 수정하지 않고 이 준비 셀에서 직접 등록한다.
        prepared_product = supabase.table("products").insert({
            "seller_id": seller_id,
            "name": "EX 발표용 머그컵",
            "price": 12000,
            "description": "주문 연결을 보여주기 위한 실습용 상품",
        }).execute().data
        if not prepared_product:
            raise RuntimeError("상품 응답이 비었습니다. Table Editor에서 저장 여부를 먼저 확인하세요.")
        pprint([{k: row[k] for k in ("id", "name", "price")} for row in prepared_product])
    finally:
        seller_password = None
        supabase.auth.sign_out()
else:
    print("판매자 준비를 건너뜁니다. 기존 상품으로 진행할 수 있습니다.")

## 1. 회원가입 폼: 입력값을 받고 Auth 계정 생성

GUI라면 이메일 입력창, 비밀번호 입력창, 회원가입 버튼이 있는 화면이다. 여기서는 `input()`, `getpass()`, 코드 셀 실행으로 대신한다.

`CREATE_ACCOUNT = True`일 때만 새 계정을 생성한다. 이미 가입한 계정으로 발표할 때는 `False`로 둔다. 가입 셀을 반복 실행하는 대신 같은 계정으로 로그인하면 된다.

`auth.sign_up()`은 Supabase Auth에 가입을 요청한다. `public.user_details`에 주소를 넣는 작업과 구분해야 한다. DB에 별도 트리거가 없다면 상세 정보는 다음 단계에서 직접 만든다.

이메일 확인 설정이 켜져 있으면 가입 응답의 `session`이 없을 수 있다. 이것은 로그인 완료가 아니다. 메일 인증 후 `CREATE_ACCOUNT=False`로 다시 입력하고 로그인한다. 비밀번호/토큰이 들어 있는 응답 객체 전체를 출력하지 않는다.

In [ ]:
CREATE_ACCOUNT = False  # 처음 계정을 만들 때만 True
email = input("구매자 이메일: ").strip()
password = getpass("구매자 비밀번호: ")
if not email or not password:
    raise ValueError("이메일과 비밀번호를 입력하세요.")

if CREATE_ACCOUNT:
    signup = supabase.auth.sign_up({"email": email, "password": password})
    if signup.session is None:
        password = None
        raise RuntimeError("가입 이메일을 확인한 뒤 CREATE_ACCOUNT=False로 이 셀부터 다시 실행하세요.")
    print("가입 요청 완료. 다음 셀에서 로그인을 확인합니다.")
else:
    print("기존 계정으로 로그인할 준비가 되었습니다.")

## 2. 로그인 버튼: 실제 인증된 사용자 ID 받기

이메일과 비밀번호로 로그인하고 서버에서 현재 사용자를 다시 확인한다. `user_id`는 임의로 만든 UUID가 아니라 **로그인한 계정의 실제 ID**다. 이 값을 회원 상세 정보와 주문의 연결 기준으로 사용한다.

비밀번호는 로그인 요청 후 변수에서 비운다. 셀 실패 후 재시도하려면 1단계에서 비밀번호를 다시 입력한다. 변수에 `None`을 넣는 것이 메모리의 모든 복사본을 완전히 지운다는 뜻은 아니다.

In [ ]:
# 로그인 재시도가 실패했을 때 이전 사용자 ID로 진행하지 않도록 먼저 비운다.
user_id = None
try:
    login = supabase.auth.sign_in_with_password({"email": email, "password": password})
    if login.user is None or login.session is None:
        raise RuntimeError("로그인이 완료되지 않았습니다.")
    current_user = supabase.auth.get_user()
    if current_user.user is None:
        raise RuntimeError("현재 로그인 사용자를 확인할 수 없습니다.")
    user_id = current_user.user.id
finally:
    password = None

print("로그인 성공. 이후 주문에는 현재 사용자의 ID가 들어갑니다.")

## 3. 회원 상세 정보: Auth와 1:1로 연결

`auth.users`가 인증 계정을 저장한다면, `user_details`는 BUYER/SELLER 유형과 주소 등 서비스 정보를 저장한다. 두 테이블의 `id`를 같게 넣어 어떤 계정의 상세 정보인지 연결한다.

먼저 내 ID로 조회한다. 이미 있다면 재사용하고 없을 때만 `create_user_detail()`을 호출한다. 가입 시 자동 생성하는 DB 트리거가 있는 경우에도 중복 생성을 피할 수 있다. 소프트 삭제된 회원은 자동으로 복구하지 않는다.

아래 주소는 설명용 문자열이다. 실제 개인정보를 발표 화면에 쓰지 않아도 된다. `update_user_detail()`은 전달한 필드만 수정한다. `**kwargs`는 이름이 붙은 여러 인자를 딕셔너리처럼 받는 Python 문법이다.

In [ ]:
if not user_id:
    raise RuntimeError("2단계 로그인을 먼저 완료하세요.")

details = supabase.table("user_details").select("*").eq("id", user_id).execute().data
if not details:
    details = user_service.create_user_detail(user_id, type="BUYER")
if not details or details[0]["deleted_at"] is not None:
    raise RuntimeError("활성 회원 상세 정보가 필요합니다. 저장 여부와 권한을 확인하세요.")
if details[0]["type"] != "BUYER":
    raise PermissionError("본 시연은 BUYER 계정으로 진행합니다. 구매자 계정으로 로그인하세요.")

# GUI의 배송지 입력창을 대신한다. 필요하면 발표용 값만 수정한다.
delivery = {"zipcode": "00000", "address": "발표용 가상 주소", "address_sub": "실습실"}
updated_details = user_service.update_user_detail(user_id, **delivery)
if not updated_details:
    raise RuntimeError("회원 정보 수정 결과가 없습니다. 현재 계정의 RLS 권한을 확인하세요.")
print("회원 상세 정보 준비 완료: BUYER / 발표용 배송지")

## 4. 상품 목록과 선택: DB 응답을 화면 데이터로 사용

`select()`는 읽기, `is_("deleted_at", "null")`은 삭제되지 않은 행만 고르는 조건이다. Python의 `is`와 함수 이름이 충돌하지 않도록 SDK 이름에 밑줄이 붙어 있다.

`execute().data`는 행을 담은 **리스트**다. 각 원소는 컬럼 이름을 키로 갖는 딕셔너리다. 따라서 한 상품은 `products[0]`, 그 상품 ID는 `products[0]["id"]`로 읽는다. `response.data["id"]`처럼 리스트에 문자열 키를 쓰면 오류다.

GUI에서는 상품 카드와 수량 입력창을 보여주겠지만, 여기서는 목록 번호와 변수로 대신한다. 고객이 임의로 입력한 가격을 주문 금액으로 쓰지 않도록, 주문 직전 선택한 상품을 다시 조회한다. 실제 서비스의 최종 금액 검증은 서버에서 해야 한다.

In [ ]:
products = (supabase.table("products")
    .select("id,name,price")
    .is_("deleted_at", "null")
    .order("name")
    .limit(20)
    .execute().data)
if not products:
    raise RuntimeError("보이는 상품이 없습니다. 선택 단계 S에서 상품을 준비하거나 상품 SELECT 정책을 확인하세요.")
for number, product in enumerate(products, start=1):
    print(f"{number}. {product['name']} / {product['price']}원 / ID={product['id']}")

In [ ]:
product_number = int(input("주문할 상품 번호: "))
quantity = 2  # GUI의 수량 입력창. 1, 2, 3 등 양의 정수로 수정한다.
if not 1 <= product_number <= len(products):
    raise ValueError("목록에 있는 상품 번호를 선택하세요.")
if type(quantity) is not int or quantity <= 0:
    raise ValueError("수량은 1 이상의 정수여야 합니다.")
selected_product_id = products[product_number - 1]["id"]
print("선택 완료. 아직 주문을 저장하지 않았습니다.")

## 5. 주문하기 버튼: 두 테이블을 순서대로 생성

이 셀 한 번 실행이 GUI의 **주문하기 버튼 한 번 클릭**이다. 핵심 흐름은 다음과 같다.

1. 선택한 상품의 현재 이름과 가격을 읽는다.
2. `가격 × 수량`으로 주문 총액을 계산한다.
3. `orders`에 주문번호, 주문자 표시명, 사용자 ID, 배송지, 총액을 넣어 주문 한 건을 만든다.
4. 생성 결과에서 `orders.id`를 받는다.
5. `order_items`에 그 주문 ID와 선택한 상품 ID를 함께 넣는다.

`orders`는 영수증 한 장, `order_items`는 영수증에 적힌 구매 항목, `products`는 상품 원본이다. 주문에 상품이 여러 종류 들어가면 같은 `order_id`로 주문상품 행을 여러 개 만들면 된다. 오늘 예제는 상품 한 종류만 주문한다.

**FK가 두 번째 행을 자동으로 만들어 주지는 않는다.** FK는 연결 대상의 존재를 검사한다. 위 순서로 메서드를 호출하는 `ex`의 코드가 자동화 역할을 맡는다.

`item_name`, `item_price`에는 주문 당시 상품명과 단가를 복사한다. 나중에 상품 가격이 바뀌더라도 구매 당시 가격을 설명할 수 있게 하는 스냅샷이다. `products.price`는 현재 가격, `order_items.item_price`는 주문 당시 가격이다.

`Decimal(str(price))`는 가격을 십진수로 계산하기 위한 것이다. 예제는 원 단위 정수 금액만 허용하고 JSON에 저장할 때 `int`로 바꾼다. 복잡한 결제 시스템이나 통화 라이브러리는 추가하지 않는다.

실제 `orders` 정의에서 `order_no`와 `order_name`은 기본값이 없는 필수 컬럼이다. `order_no`는 UNIQUE이므로 매 주문마다 UUID로 새 번호를 만든다. DB 내부 관계에는 `id`, 사람이 보는 주문번호에는 `order_no`를 사용한다. `order_status`는 기본값과 CHECK 조건이 없는 nullable text다. 예제에서는 `PENDING`을 주문 접수 상태 문자열로 명시한다. 미리 정해진 DB enum을 읽어 온 값은 아니다.

**실패와 재실행:** 두 `insert` 요청은 하나의 트랜잭션이 아니다. 주문 생성 후 주문상품 생성이 실패하면 주문만 남을 수 있다. 그래서 주문 ID를 먼저 보관하고 7단계에서 해당 주문을 정리할 수 있게 한다. 주문 ID가 남아 있으면 중복 주문을 막기 위해 이 셀을 멈춘다. 네트워크 오류로 저장 결과가 불분명할 때도 먼저 DB를 확인한다. 이 변수 검사는 서버의 중복 요청 방지 기능을 대체하지 않는다.

In [ ]:
if not user_id:
    raise RuntimeError("먼저 로그인하세요.")
if demo_order_id is not None:
    raise RuntimeError("이 커널에 진행 중인 주문이 있습니다. 6단계 조회와 7단계 삭제를 먼저 실행하세요.")

selected = (supabase.table("products").select("id,name,price")
    .eq("id", selected_product_id).is_("deleted_at", "null").execute().data)
if len(selected) != 1:
    raise RuntimeError("선택한 상품이 삭제되었거나 조회할 수 없습니다.")
product = selected[0]
unit_price = Decimal(str(product["price"]))
if not unit_price.is_finite() or unit_price < 0 or unit_price != unit_price.to_integral_value():
    raise ValueError("이 예제는 0 이상의 원 단위 정수 가격만 사용합니다.")
if type(quantity) is not int or quantity <= 0:
    raise ValueError("수량은 1 이상의 정수여야 합니다.")
total_price = int(unit_price) * quantity

order_data = {
    "order_no": f"EX-{uuid.uuid4().hex}",  # NOT NULL + UNIQUE인 주문번호
    "order_name": "발표용 구매자",        # 기본값 없는 필수 표시명
    "user_id": user_id,
    "total_price": total_price,
    "order_status": "PENDING",           # 예제에서 정한 주문 접수 상태
    **delivery,
}
created_orders = order_service.create_order(order_data)
if not created_orders:
    raise RuntimeError("주문 응답이 비었습니다. 재시도 전에 Table Editor에서 저장 여부를 확인하세요.")
demo_order_id = created_orders[0]["id"]  # 부모 ID를 받아 자식 행에 전달한다.
print("생성한 주문 ID:", demo_order_id)

created_items = order_item_service.create_order_item(
    order_id=demo_order_id,
    product_id=product["id"],
    item_name=product["name"],
    item_price=int(unit_price),
    quantity=quantity,
)
if not created_items:
    raise RuntimeError("주문상품 응답이 비었습니다. 6단계에서 확인하고 필요하면 7단계로 정리하세요.")
print(f"주문 완료: {product['name']} × {quantity}개 = {total_price:,}원")
pprint(created_items)

## 6. 주문 상세 화면: 연결된 행을 한 번에 조회

현재 주문 ID와 로그인한 사용자 ID를 모두 조건으로 사용한다. `select()` 안의 `order_items(...products(...))`는 DB의 외래 키 관계를 따라 관련 데이터를 함께 가져오는 Supabase/PostgREST 문법이다. C++의 객체 포인터를 따라가는 것처럼 볼 수 있지만 실제로는 DB의 ID 관계를 이용한 API 요청이다.

주문은 딕셔너리 한 개로, 그 안의 `order_items`는 여러 구매 항목이 될 수 있으므로 리스트로 반환된다. 각 구매 항목의 `products`에는 연결된 상품 정보가 들어간다. 삭제된 주문상품은 조회 조건에서 제외한다.

Table Editor에서 확인할 값은 `orders.id == order_items.order_id`, `products.id == order_items.product_id`다. ID 문자열이 연결되었다는 것과 실제 FK 제약이 설정되어 있다는 것은 구분해야 한다. 아래 관계 조회는 작성 시 빈 결과 조회로 구문이 받아들여짐을 확인했다.

In [ ]:
if not demo_order_id or not user_id:
    raise RuntimeError("로그인과 주문 생성이 먼저 필요합니다.")
order_view = (supabase.table("orders")
    .select("id,order_no,order_name,user_id,total_price,order_status,deleted_at,order_items(id,order_id,product_id,item_name,item_price,quantity,deleted_at,products(id,name,price))")
    .eq("id", demo_order_id)
    .eq("user_id", user_id)
    .is_("deleted_at", "null")
    .is_("order_items.deleted_at", "null")
    .execute().data)
if not order_view:
    raise RuntimeError("현재 사용자에게 보이는 활성 주문이 없습니다. 주문 ID와 권한을 확인하세요.")
pprint(order_view)

## 7. 주문 삭제 버튼: 주문상품과 주문을 함께 소프트 삭제

여기서 삭제는 `DELETE FROM`으로 행을 없애는 것이 아니라 `deleted_at`에 현재 시각을 넣는 **UPDATE**다. 그래서 일반 조회에서는 숨겨지고, Table Editor에는 기록이 남는다. `deleted_at IS NULL`이면 활성, `IS NOT NULL`이면 삭제된 기록이다.

부모 주문의 `deleted_at`만 바꾼다고 주문상품에도 같은 값이 자동으로 들어가지는 않는다. `ON DELETE CASCADE`는 실제 DELETE의 규칙이며 이 UPDATE를 자동 전파하지 않는다. 아래 셀은 내 주문 확인 → 활성 주문상품들 소프트 삭제 → 부모 주문 소프트 삭제 순서로 기존 메서드를 호출한다.

**상품 원본은 삭제하지 않는다.** 다른 주문에서도 같은 상품을 참조할 수 있기 때문이다. 이 단계는 주문 기록을 숨기는 예제이며 환불, 결제 취소, 재고 복구는 구현하지 않는다.

도중에 실패하면 성공한 UPDATE는 DB에 남는다. 실패한 자식이 있는데 부모까지 삭제됐다고 표시하지 않도록 결과를 검사한다. 셀을 재실행하면 이미 삭제된 항목은 제외하고 남은 작업을 진행한다. 실제 서비스에서 여러 요청을 한꺼번에 성공/실패시켜야 한다면 서버 트랜잭션이 필요하지만 오늘 범위에는 추가하지 않는다.

확인한 대시보드에서는 `orders`와 `order_items`의 RLS가 비활성화되어 있었다. 아래 내 주문 검사는 예제 코드의 동작 조건이지 서버 권한 정책을 구축했다는 뜻은 아니다. 이번에는 DB 정책을 변경하지 않는다.

In [ ]:
if not demo_order_id or not user_id:
    raise RuntimeError("삭제할 주문 ID와 로그인 사용자가 필요합니다.")

# order_items에는 user_id가 없으므로 먼저 부모 주문이 내 것인지 확인한다.
owned_order = (supabase.table("orders").select("id,deleted_at")
    .eq("id", demo_order_id).eq("user_id", user_id).execute().data)
if not owned_order:
    raise PermissionError("내 주문인지 확인할 수 없어 삭제하지 않습니다.")

active_items = order_item_service.get_order_items(demo_order_id)
for item in active_items:
    deleted_items = order_item_service.soft_delete_order_item(item["id"])
    if not deleted_items:
        raise RuntimeError("주문상품 삭제가 확인되지 않았습니다. 권한/저장 결과 확인 후 다시 실행하세요.")

if owned_order[0]["deleted_at"] is None:
    deleted_orders = order_service.delete_order(user_id, demo_order_id)
    if not deleted_orders:
        raise RuntimeError("부모 주문 삭제가 확인되지 않았습니다. 주문 ID를 유지하고 다시 확인하세요.")

print("주문과 주문상품의 소프트 삭제 요청을 마쳤습니다. 다음 셀에서 결과를 확인합니다.")

## 8. 삭제 결과 확인과 로그아웃

삭제 요청이 오류 없이 끝났다는 것만으로 발표를 마치지 않고 다시 조회한다. 활성 주문/주문상품은 없어야 하고, 삭제된 기록에는 시각이 남아 있어야 한다. 상품은 그대로 남아 있어야 한다.

검증이 끝나야 `demo_order_id`를 비워 다음 시연을 시작할 수 있게 한다. 삭제 전에 커널을 재시작했다면 DB에서 **이번 시연의 내 주문 ID**를 확인하여 `demo_order_id`에 넣고 로그인한 뒤 7단계부터 정리한다. 이름이 비슷한 다른 사람의 주문 ID를 사용하지 않는다.

In [ ]:
if not demo_order_id or not user_id:
    raise RuntimeError("검증할 주문 ID와 로그인 사용자가 필요합니다.")
deleted_order = (supabase.table("orders").select("id,deleted_at")
    .eq("id", demo_order_id).eq("user_id", user_id).execute().data)
active_items = order_item_service.get_order_items(demo_order_id)
deleted_items = order_item_service.get_deleted_order_items(demo_order_id)
if not deleted_order or deleted_order[0]["deleted_at"] is None or active_items:
    raise RuntimeError("삭제 검증이 끝나지 않았습니다. 7단계 결과와 DB 권한을 확인하세요.")

print("부모 주문 삭제 시각:", deleted_order[0]["deleted_at"])
print("남은 활성 주문상품:", len(active_items))
print("삭제 기록으로 남은 주문상품:", len(deleted_items))

# 주문상품 생성이 실패한 주문을 정리했다면 deleted_items는 0개일 수 있다.
# 정상 시연에서는 위에서 삭제한 주문상품의 product_id로 원본 상품을 확인한다.
for item in deleted_items:
    original = (supabase.table("products").select("id,name,deleted_at")
        .eq("id", item["product_id"]).execute().data)
    if not original or original[0]["deleted_at"] is not None:
        raise RuntimeError("상품 원본을 활성 상태로 확인할 수 없습니다. DB 상태와 권한을 확인하세요.")
    print("상품 원본 유지:", original[0]["name"])

last_demo_order_id = demo_order_id
demo_order_id = None
print("삭제 결과 확인 완료. 다음 주문 예제를 실행할 수 있습니다.")

In [ ]:
supabase.auth.sign_out()
user_id = None
password = None
print("로그아웃 완료. 다시 주문하려면 1~3단계부터 로그인과 회원 정보를 준비하세요.")

## 발표 대본: 3~5분 설명 순서

**시작:** “저희는 회원, 상품, 주문, 주문상품을 테이블별 클래스로 나눴습니다. 저는 GUI에서 버튼을 눌렀을 때 이 기능들이 어떻게 연결되는지를 `ex.ipynb`로 보여드리겠습니다.”

**회원:** “회원가입과 로그인은 Supabase Auth가 처리합니다. 로그인 결과로 받은 사용자 ID를 회원 상세 정보에도 사용합니다. Auth에는 인증 정보, user_details에는 회원 유형과 배송지를 나눠 저장합니다.”

**상품 선택:** “GUI의 상품 목록 대신 조회 결과를 출력합니다. 사용자는 번호와 수량만 고르고, 단가는 선택한 상품을 다시 조회해서 가져옵니다.”

**주문 생성:** “주문하기 셀을 실행하면 orders를 먼저 생성합니다. 반환된 주문 ID와 상품 ID를 order_items에 넣으면 누가 어떤 상품을 몇 개 주문했는지 연결됩니다. FK가 행을 자동 생성하는 것이 아니라, Python 코드가 두 저장 작업을 순서대로 호출합니다.”

**관계 조회:** “주문 안의 order_items와 그 항목이 가리키는 products를 함께 조회했습니다. 주문은 하나여도 주문상품은 여러 개가 될 수 있기 때문에 별도 테이블로 분리했습니다. item_price에는 구매 당시 단가를 보관합니다.”

**삭제:** “삭제 버튼도 셀로 대신합니다. 주문상품과 주문의 deleted_at을 차례로 갱신해서 활성 조회에서 제외합니다. 상품 원본은 남습니다. 이것은 기록의 소프트 삭제이며 실제 결제 취소는 아닙니다.”

**한계:** “이번 예제의 두 저장 요청은 하나의 트랜잭션이 아닙니다. 일부만 성공할 수 있어서 생성한 주문 ID를 보관하고 정리할 수 있도록 했습니다. 원자적인 주문 처리와 실제 GUI는 다음 단계로 남겨 두었습니다.”

## 자주 묻는 질문과 오류 읽기

| 질문/증상 | 설명과 확인 순서 |
|---|---|
| 이게 프론트엔드인가요? | GUI 자체는 아니다. 프론트엔드의 입력과 버튼 이벤트를 셀로 대신하고, 여러 CRUD를 연결하는 응용 흐름까지 `ex`에서 보여준다. |
| 클래스 하나 호출하면 모든 테이블이 바뀌나요? | 각 메서드는 담당 쿼리를 실행한다. `ex`가 여러 메서드를 순서대로 호출해 한 사용자 동작으로 묶는다. |
| 상품 가격을 주문상품에 왜 또 저장하나요? | 현재 판매 가격과 과거 구매 가격은 의미가 다르다. 주문 시점 단가를 보존한다. |
| `execute()`는 왜 필요한가요? | select/eq/insert 등은 요청 내용을 구성하고 execute에서 서버에 요청하여 결과를 받는다. |
| 이메일 인증 필요/이미 가입한 이메일 | 메일 인증을 완료하고 신규 가입 옵션을 False로 바꾼 뒤 로그인한다. 오류가 났다고 새 계정을 계속 만들지 않는다. |
| 401 또는 invalid API key | 프로젝트 URL과 공개 키의 짝, .env 재로드, 현재 로그인 상태를 확인한다. 키 자체를 출력하지 않는다. |
| 23502, not-null violation | 필수 컬럼을 확인한다. 현재 주문에서는 order_no와 order_name, 주문상품에서는 order_id, product_id, item_name, item_price, quantity를 빠뜨리면 안 된다. |
| 23503, foreign key violation | user_id, order_id, product_id가 실제 부모 행을 가리키는지와 생성 순서를 확인한다. |
| 23505, duplicate key | 이미 존재하는 회원 상세 정보나 행을 다시 생성했는지 확인한다. 기존 회원을 지우지 않는다. |
| 23514, check violation | 회원 유형, 주문 상태, 가격/수량의 실제 CHECK 조건을 확인한다. |
| 42501 또는 빈 응답 | SELECT/INSERT/UPDATE RLS 정책과 로그인 계정을 확인한다. 빈 응답만으로 저장 실패나 데이터 부재를 단정하지 않는다. |
| PGRST200, 관계 조회 오류 | 실제 FK와 스키마 캐시를 확인한다. 컬럼명만 같아도 자동으로 관계가 생기는 것은 아니다. |
| 주문만 남았어요 | 주문상품 저장이 실패한 상태일 수 있다. 보관한 demo_order_id를 확인하고 7~8단계로 정리한다. try/except만으로 DB가 자동 롤백되지는 않는다. |

**실제 시연 점검:** 판매 상품 준비 → 구매자 가입/인증 → 로그인 → 상세 정보 → 주문 생성 → 관계 조회 → 소프트 삭제 → 결과 재조회. 실제 실행에 성공한 단계만 성공했다고 발표한다.

## 작성 및 검증 기록, 2026-09-11

- [발표용 Notion, AX_TeamDB 발표](https://app.notion.com/p/3d8dc8b0e1638100ab71d26bd7b39870): 역할 구분, 테이블 관계도, 핵심 코드와 3~5분 대본을 한 페이지에 정리했다.
- 노트북 형식과 코드 문법, 원본 클래스 파일 네 개의 바이트 단위 보존을 확인했다.
- 모의 DB 테스트 10개로 정상 주문/삭제, 가격 스냅샷, 중복 클릭, 부분 실패 후 정리, 삭제 재시도, 다른 사용자 주문 보호, 회원 상세 재사용, 수량 검증, 인증 분기와 판매자 준비를 확인했다.
- 실제 DB에서는 테이블 정의/정책을 조회하고 중첩 관계 조회 구문을 확인했다. 실제 회원가입, 로그인, 주문 쓰기는 실행하지 않았다.
- 남은 실행 전제는 상품 RLS 정책 준비와 실제 계정을 사용한 시연이다. 기존 DB 데이터와 권한은 변경하지 않았다.